In [1]:
def readctm(filename):
    lines = []
    with open(filename) as inf:
        for line in inf.readlines():
            lines.append(line.strip().split(" "))
    return lines

In [37]:
def strctm(text):
    return [line.strip().split(" ") for line in text.split("\n") if line != ""]

In [2]:
def normword(word):
    if word.endswith("..."):
        word = word[0:-3]
    if word[-1:] in ".,?!-:;":
        word = word[0:-1]
    return word.lower()

In [32]:
def push_back_ins(ctm_lines, start, end, offset):
    pre = []
    post = []
    buf = []

    for line in ctm_lines:
        if float(line[2]) < start:
            pre.append(line)
        elif float(line[2]) <= end:
            buf.append(line)
        else:
            post.append(line)

    i = 0
    while i < (len(buf) - offset):
        print(buf[i], buf[i+offset])
        replace = buf[i+offset][6]
        replacenorm = normword(replace)
        buf[i][4] = replacenorm
        buf[i][6] = replace
        buf[i][7] = "cor"
        i += 1
    while i < len(buf):
        buf[i][6] = "-"
        buf[i][7] = "-"
        i += 1
        
    return pre + buf + post

In [56]:
def merge_left(ctmlines, replacement = None):
    outword = ctmlines[-1][6]
    if replacement is not None:
        outword = replacement
    out = ctmlines[0][0:3]
    out += [ctmlines[-1][3], normword(outword), "1.0", outword, "cor"]
    return out

In [33]:
ctmlines = readctm("/tmp/ctmedit/hsi_1_0515_209_001_inter.txt")

In [60]:
def printctm(toprint):
    for ll in toprint:
        print(" ".join(ll))

In [67]:
def as_csv(c):
    print(f"{c[2]}\t{c[3]}\t{c[6]}")

In [106]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "sub":
            outlines.append(ctmline)
        else:
            if ctmline[4] == normword(ctmline[6]):
                ctmline[7] = "cor"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

80008 cor
22110 sub
11758 ins
7578 del

In [75]:
from pathlib import Path

CTMDIR = Path("/Users/joregan/Playing/hsi_ctmedit")

for ctmfile in CTMDIR.glob("*.txt"):
    ctmlines = readctm(str(ctmfile))
    outlines = []
    for ctmline in ctmlines:
        if len(ctmline) < 7:
            outlines.append(ctmline)
            continue
        if ctmline[7] != "ins":
            outlines.append(ctmline)
        else:
            if ctmline[4] in ["ah", "eh", "oh", "mhm", "m", "um"]:
                ctmline[7] = "cor"
                ctmline[6] = f"[{ctmline[4]}]"
            outlines.append(ctmline)
    with open(str(ctmfile), "w") as outf:
        for line in outlines:
            outf.write(" ".join(line) + "\n")

In [90]:
import json
from pathlib import Path
def json_ctm(filename, edit = False):
    with open(filename) as inf:
        data = json.load(inf)
        fileid = Path(filename).stem.replace(".w2v", "")
        for chunk in data['chunks']:
            start = chunk['timestamp'][0]
            end = chunk['timestamp'][1]
            word = chunk['text'].lower()
            if edit:
                edit = f" {word} cor"
            else:
                edit = ""
            print(f"{fileid} 1 {start} {end} {word} 1.0{edit}")

In [ ]:
json_ctm('/Users/joregan/Playing/hsi/audio/wav2vec/hsi_7_0719_222_003_main.w2v.json')

In [88]:
def json_ctm_path(fid, ctmedit=True):
    return json_ctm(f"/Users/joregan/Playing/hsi/audio/wav2vec/{fid}.w2v.json", ctmedit)

In [127]:
json_ctm_path("hsi_5_0718_209_002_inter", True)

hsi_5_0718_209_002_inter 1 1.04 1.18 but 1.0 but cor
hsi_5_0718_209_002_inter 1 1.42 1.44 a 1.0 a cor
hsi_5_0718_209_002_inter 1 1.58 1.66 ye 1.0 ye cor
hsi_5_0718_209_002_inter 1 1.78 1.8 i 1.0 i cor
hsi_5_0718_209_002_inter 1 1.86 1.98 think 1.0 think cor
hsi_5_0718_209_002_inter 1 2.04 2.1 the 1.0 the cor
hsi_5_0718_209_002_inter 1 2.14 2.32 next 1.0 next cor
hsi_5_0718_209_002_inter 1 2.42 2.5 one 1.0 one cor
hsi_5_0718_209_002_inter 1 2.56 2.72 would 1.0 would cor
hsi_5_0718_209_002_inter 1 2.78 2.94 be 1.0 be cor
hsi_5_0718_209_002_inter 1 3.72 3.9 maybe 1.0 maybe cor
hsi_5_0718_209_002_inter 1 3.94 4.0 you 1.0 you cor
hsi_5_0718_209_002_inter 1 4.04 4.14 can 1.0 can cor
hsi_5_0718_209_002_inter 1 4.2 4.34 put 1.0 put cor
hsi_5_0718_209_002_inter 1 4.42 4.46 in 1.0 in cor
hsi_5_0718_209_002_inter 1 4.58 4.6 a 1.0 a cor
hsi_5_0718_209_002_inter 1 4.76 5.26 teapos 1.0 teapos cor
hsi_5_0718_209_002_inter 1 6.54 6.76 now 1.0 now cor
hsi_5_0718_209_002_inter 1 7.88 8.12 that's 1.0 tha

In [122]:
def whisperx_json_ctm(filepath, ctmedit=False):
    with open(filepath) as inf:
        data = json.load(inf)
    fileid = Path(filepath).stem
    for segment in data['segments']:
        for word in segment['words']:
            if "speaker" in word:
                speaker = word['speaker']
                channel = int(speaker.replace("SPEAKER_0", "")) + 1
                start = word['start']
                end = word['end']
            else:
                channel = 1
                start = 0.0
                end = 0.0
            text = word['word']
            conf = word['score'] if "score" in word else 0.0
            norm = normword(text)
            if ctmedit:
                edit = f" {text} cor"
            else:
                edit = ""
            print(f"{fileid} {channel} {start} {end} {norm} {conf}{edit}")

In [97]:
def my_whisperx_json_ctm(a, b=True):
    whisperx_json_ctm(f"/Users/joregan/Playing/hsi/audio/whisperx-json/{a}.json", b)

In [124]:
my_whisperx_json_ctm("hsi_5_0718_209_002_inter")

hsi_5_0718_209_002_inter 1 1.049 1.21 but 0.999 But cor
hsi_5_0718_209_002_inter 1 1.59 1.75 yeah 0.607 yeah, cor
hsi_5_0718_209_002_inter 1 1.79 1.83 i 1.0 I cor
hsi_5_0718_209_002_inter 1 1.87 2.01 think 0.899 think cor
hsi_5_0718_209_002_inter 1 2.03 2.11 the 0.994 the cor
hsi_5_0718_209_002_inter 1 2.15 2.33 next 0.842 next cor
hsi_5_0718_209_002_inter 1 2.41 2.53 one 0.748 one cor
hsi_5_0718_209_002_inter 1 2.55 2.73 could 0.878 could cor
hsi_5_0718_209_002_inter 1 2.79 3.01 be 0.876 be cor
hsi_5_0718_209_002_inter 1 3.711 3.911 maybe 0.953 maybe cor
hsi_5_0718_209_002_inter 1 3.931 4.011 you 0.837 you cor
hsi_5_0718_209_002_inter 1 4.051 4.171 can 0.931 can cor
hsi_5_0718_209_002_inter 1 4.191 4.351 put 0.939 put cor
hsi_5_0718_209_002_inter 1 4.411 4.491 in 0.907 in cor
hsi_5_0718_209_002_inter 1 4.591 4.651 a 0.848 a cor
hsi_5_0718_209_002_inter 1 4.751 5.312 t-pose 0.843 t-pose cor
hsi_5_0718_209_002_inter 1 6.552 6.792 now 1.0 now cor
hsi_5_0718_209_002_inter 1 7.893 8.133 th

In [107]:
def ctmedit_to_tsv(fileid):
    filename = f"/Users/joregan/Playing/hsi_ctmedit/{fileid}.txt"
    outfile = f"/tmp/{fileid}.tsv"
    with open(filename) as inf, open(outfile, "w") as outf:
        lastend = 0.0
        for line in inf.readlines():
            line = line.strip()
            if line == "":
                outf.write("\n")
                continue
            parts = line.split()
            lastend = float(parts[3])
            if line.endswith(" cor"):
                outf.write(f"{parts[2]}\t{parts[3]}\t{parts[6]}\n")
            elif line.endswith(" sub"):
                outf.write(f"{parts[2]}\t{parts[3]}\t{parts[4]}/{parts[6]}\n")
            elif line.endswith(" ins"):
                outf.write(f"{parts[2]}\t{parts[3]}\t[{parts[4]}]\n")
            elif line.endswith(" del"):
                outf.write(f"{lastend}\t{lastend + 0.1}\t[{parts[4]}]\n")

In [126]:
ctmedit_to_tsv("hsi_5_0718_209_002_inter")

In [115]:
def set_to_left(text):
    lines = text.split("\n")
    for line in lines:
        if line == "":
            continue
        parts = line.split(" ")
        parts[6] = parts[4]
        parts[7] = "cor"
        print(" ".join(parts))

In [116]:
text = """
hsi_5_0718_209_001_inter 1 209.64 209.82 can 1.0 do sub
hsi_5_0718_209_001_inter 1 209.82 209.91 you 1.0 you cor
hsi_5_0718_209_001_inter 1 209.91 210.3 just 1.0 - ins
hsi_5_0718_209_001_inter 1 210.6 211.05 expand 1.0 - ins
hsi_5_0718_209_001_inter 1 211.05 211.2 with 0.664748 - ins
hsi_5_0718_209_001_inter 1 211.2 211.38 about 1.0 - ins
hsi_5_0718_209_001_inter 1 211.38 211.5 your 1.0 - ins
hsi_5_0718_209_001_inter 1 211.5 212.19 paintings 1.0 - ins
"""
set_to_left(text)

hsi_5_0718_209_001_inter 1 209.64 209.82 can 1.0 can cor
hsi_5_0718_209_001_inter 1 209.82 209.91 you 1.0 you cor
hsi_5_0718_209_001_inter 1 209.91 210.3 just 1.0 just cor
hsi_5_0718_209_001_inter 1 210.6 211.05 expand 1.0 expand cor
hsi_5_0718_209_001_inter 1 211.05 211.2 with 0.664748 with cor
hsi_5_0718_209_001_inter 1 211.2 211.38 about 1.0 about cor
hsi_5_0718_209_001_inter 1 211.38 211.5 your 1.0 your cor
hsi_5_0718_209_001_inter 1 211.5 212.19 paintings 1.0 paintings cor
